# OmniVoice Project Studio — Kaggle Gradio Simple (Dual-T4 optimized)

Minimal Gradio-only launcher tuned for Kaggle **2× Tesla T4** sessions.

```text
cuda:0  → OmniVoice TTS
cuda:1  → Whisper ASR verification
CPU/RAM → preprocessing + Gradio + file I/O
SSD     → /kaggle/working/OmniVoiceStudio + model cache
```

If Kaggle exposes only one GPU, the notebook automatically keeps OmniVoice on `cuda:0` and falls back to CPU for ASR. The workspace is ephemeral and disappears when the Kaggle session is discarded.


In [ ]:
# Kaggle Internet must be enabled for GitHub/Hugging Face downloads.
import os
from pathlib import Path

WORKSPACE = "/kaggle/working/OmniVoiceStudio"
CACHE_ROOT = "/kaggle/working/.cache"
Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
Path(CACHE_ROOT).mkdir(parents=True, exist_ok=True)

# Keep transient model caches on Kaggle local SSD and reduce long-run CUDA fragmentation.
os.environ["HF_HOME"] = f"{CACHE_ROOT}/huggingface"
os.environ["TORCH_HOME"] = f"{CACHE_ROOT}/torch"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@master"


In [ ]:
import shutil
import torch

from omnivoice.hardware_quality import (
    HardwareQualitySettingsStore,
    detect_hardware,
)

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU T4 x2 in Kaggle Notebook settings.")

GPU_COUNT = torch.cuda.device_count()
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory / 1024**3:.1f} GiB")

TTS_DEVICE = "cuda:0"
ASR_DEVICE = "cuda:1" if GPU_COUNT >= 2 else "cpu"
ASR_MODEL = "openai/whisper-small.en"

hardware = detect_hardware(device_index=0)
HardwareQualitySettingsStore(WORKSPACE).set_default(hardware.recommended_preset)

print("Hardware profile:", hardware.summary())
print("OmniVoice device:", TTS_DEVICE)
print("Whisper ASR device:", ASR_DEVICE)
print("Whisper ASR model:", ASR_MODEL)
print("Default quality preset:", hardware.recommended_preset)
print("Workspace:", WORKSPACE)
usage = shutil.disk_usage("/kaggle/working")
print(f"Local SSD free: {usage.free / 1024**3:.1f} GiB")


## Launch Gradio

`whisper-small.en` remains the default verifier because it is accurate enough for English exact-text checks while being very fast on the dedicated second T4. For a stricter but slower verifier, change `ASR_MODEL` to `openai/whisper-medium.en`.


In [ ]:
!omnivoice-project-studio \
  --model k2-fsa/OmniVoice \
  --device {TTS_DEVICE} \
  --workspace {WORKSPACE} \
  --asr-model {ASR_MODEL} \
  --asr-device {ASR_DEVICE} \
  --share
